In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import re


/home/tulocalhost/RETO_TC3002B.201/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/tulocalhost/RETO_TC3002B.201/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 25580.51it/s]


NameError: name 'df' is not defined

In [ ]:


# Ruta del dataset
INPUT_CSV = "/data/filtered_H-AIRosettaMP.csv"

# Función para limpiar código fuente
def clean_code(code):
    # Elimina comentarios de una sola línea (// ...)
    code = re.sub(r'//.*', '', code)

    # Elimina comentarios multilínea (/* ... */)
    code = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)

    # Normaliza espacios (muchos espacios → uno solo)
    code = re.sub(r'\s+', ' ', code)

    return code.strip()  # Quita espacios al inicio y final

# Cargar dataset
df = pd.read_csv(
    INPUT_CSV,
    encoding="latin1",
    engine="python"
)
# Aplicar limpieza al código
df["code_clean"] = df["code"].astype(str).apply(clean_code)

# Normalizar etiquetas (lowercase y sin espacios extra)
df["target"] = df["target"].astype(str).str.strip().str.lower()

# Mapear etiquetas a valores numéricos
label_map = {
    "human_written": 0,
    "ai_generated": 1
}

df["target_num"] = df["target"].map(label_map)

# Eliminar filas donde no se pudo mapear la etiqueta
df = df.dropna(subset=["target_num"])

# Mostrar distribución de clases
print("Distribución:", df["target_num"].value_counts())

# Verificar que hay exactamente 2 clases
assert df["target_num"].nunique() == 2, "Solo hay una clase"

# Extraer datos
codes = df["code_clean"].tolist()  # lista de códigos limpios
y = df["target_num"].values        # etiquetas

# Cargar modelo preentrenado CodeBERT
model = SentenceTransformer("microsoft/codebert-base")

# Convertir código a embeddings (vectores numéricos)
X = model.encode(
    codes,
    batch_size=64,
    show_progress_bar=True
)

# Convertir a array de numpy
X = np.array(X)

# División del dataset:
# 70% entrenamiento, 30% temporal (validación + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Dividir el 30% restante en:
# 15% validación y 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

# Crear modelo de regresión logística
clf = LogisticRegression(max_iter=2000)

# Entrenar el modelo con los embeddings
clf.fit(X_train, y_train)

print("Modelo entrenado")

print("Train:", X_train.shape, len(y_train))
print("Validation:", X_val.shape, len(y_val))
print("Test:", X_test.shape, len(y_test))

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    recall_score,
    accuracy_score,
    precision_score
)

def evaluate_threshold(X, y, model, threshold):
    probs = model.predict_proba(X)[:, 1]
    y_pred = (probs >= threshold).astype(int)

    cm = confusion_matrix(y, y_pred)

    acc = accuracy_score(y, y_pred)
    recall = recall_score(y, y_pred)
    precision = precision_score(y, y_pred)

    # F1 por clase
    f1_per_class = f1_score(y, y_pred, average=None)

    print(f"\nThreshold = {threshold}")
    print("Accuracy:", acc)
    print("Recall:", recall)
    print("Precision:", precision)
    print("F1 por clase [human, ai]:", f1_per_class)
    print("Confusion Matrix:\n", cm)


for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    evaluate_threshold(X_val, y_val, clf, t)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(X, y, model, title="Confusion Matrix"):
    y_pred = model.predict(X)
    cm = confusion_matrix(y, y_pred)

    plt.figure(figsize=(6,5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Human", "AI"],
        yticklabels=["Human", "AI"]
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.show()

# Ejemplo:
plot_confusion_matrix(X_test, y_test, clf, "Test Confusion Matrix")